# Day 12 - Credit Card Fraud Detection
**100 Days of ML & AI**

This notebook covers:
- Exploratory Data Analysis
- Handling Imbalanced Datasets
- Logistic Regression vs Random Forest
- Precision, Recall, ROC-AUC evaluation


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, roc_curve,
    ConfusionMatrixDisplay
)

pd.set_option('display.float_format', '{:.4f}'.format)
print('Libraries imported.')

## 1. Load Dataset

In [ ]:
df = pd.read_csv('dataset/creditcard.csv')
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head()

## 2. EDA

In [ ]:
# Class balance
counts = df['Class'].value_counts()
print(counts)
print(f"Fraud rate: {df['Class'].mean()*100:.3f}%")

fig, ax = plt.subplots(figsize=(6,4))
ax.bar(['Legitimate','Fraud'], [counts[0], counts[1]], color=['#2196F3','#F44336'])
ax.set_title('Class Distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('screenshots/class_distribution.png', dpi=100)
plt.show()

In [ ]:
# Correlation Heatmap
cols = [f'V{i}' for i in range(1,11)] + ['Amount','Class']
fig, ax = plt.subplots(figsize=(10,8))
sns.heatmap(df[cols].corr(), cmap='coolwarm', center=0, linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap (V1-V10, Amount, Class)')
plt.tight_layout()
plt.savefig('screenshots/correlation_heatmap.png', dpi=100)
plt.show()

## 3. Feature Scaling

In [ ]:
scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled']   = scaler.fit_transform(df[['Time']])

feature_cols = [f'V{i}' for i in range(1,29)] + ['Amount_scaled','Time_scaled']
X = df[feature_cols]
y = df['Class']
print('Features:', len(feature_cols))

## 4. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print('Train:', X_train.shape, '| Test:', X_test.shape)

## 5. Train Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1),
}

results = {}
for name, clf in models.items():
    print(f'Training {name} ...')
    clf.fit(X_train, y_train)
    y_pred  = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:,1]
    results[name] = dict(
        model=clf, y_pred=y_pred, y_proba=y_proba,
        accuracy  = accuracy_score(y_test, y_pred),
        precision = precision_score(y_test, y_pred),
        recall    = recall_score(y_test, y_pred),
        f1        = f1_score(y_test, y_pred),
        roc_auc   = roc_auc_score(y_test, y_proba),
    )
    r = results[name]
    print(f'  Accuracy={r["accuracy"]:.4f}  Precision={r["precision"]:.4f}  Recall={r["recall"]:.4f}  F1={r["f1"]:.4f}  AUC={r["roc_auc"]:.4f}')

## 6. Visualizations

In [ ]:
best = results['Random Forest']

# Confusion Matrix
fig, ax = plt.subplots(figsize=(6,5))
cm = confusion_matrix(y_test, best['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legitimate','Fraud'],
            yticklabels=['Legitimate','Fraud'], ax=ax)
ax.set_title('Confusion Matrix - Random Forest')
ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('screenshots/confusion_matrix.png', dpi=100)
plt.show()

In [ ]:
# ROC Curve
fig, ax = plt.subplots(figsize=(7,5))
for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_test, r['y_proba'])
    ax.plot(fpr, tpr, label=f"{name} (AUC={r['roc_auc']:.3f})")
ax.plot([0,1],[0,1],'k--',label='Baseline')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve')
ax.legend()
plt.tight_layout()
plt.savefig('screenshots/roc_curve.png', dpi=100)
plt.show()

In [ ]:
# Feature Importance
fi = pd.Series(best['model'].feature_importances_, index=feature_cols).sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(8,5))
fi.sort_values().plot(kind='barh', color='#2196F3', ax=ax)
ax.set_title('Top 15 Feature Importances - Random Forest')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('screenshots/feature_importance.png', dpi=100)
plt.show()

In [ ]:
# Save artifacts
import os
joblib.dump(best['model'], 'model.pkl')
joblib.dump(scaler,        'scaler.pkl')
joblib.dump(feature_cols,  'feature_cols.pkl')
print('Saved model.pkl, scaler.pkl, feature_cols.pkl')

## Key Takeaways

- A naive model predicting 'Not Fraud' on everything gets 99.8% accuracy but is useless.
- **Recall** is the most important metric in fraud detection: missing a fraud is costly.
- **ROC-AUC** measures how well the model separates the two classes regardless of threshold.
- `class_weight='balanced'` compensates for the severe class imbalance automatically.
- Random Forest outperforms Logistic Regression in both Recall and AUC on this dataset.
